# Independent post-clean verification

Prove raw immutability, cleaned traceability, GDF byte identity, and CSV semantic equivalence.

Documentation authority: [Zenodo record](https://zenodo.org/records/8089820), [Scientific Data descriptor](https://www.nature.com/articles/s41597-023-02445-z), and the publisher documentation retained through `data/raw/`. Unknown semantics always force preserve-only behavior.

## Paths and safety guards

The project is a sibling of the untouched `BCI Database/`. `data/raw/` is a read-only relative symlink to that source, while `data/processed/` is the derived APFS clone.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {"notebooks", "data"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
required_directories = ("data", "models", "notebooks", "results")
if not all((PROJECT_ROOT / name).is_dir() for name in required_directories):
    raise RuntimeError("Run this notebook from bci_cleaning/ or its data/ or notebooks/ directory")

RAW_PATH = PROJECT_ROOT / "data" / "raw"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed"
REPORTS_PATH = PROJECT_ROOT / "results" / "reports"
LOGS_PATH = PROJECT_ROOT / "results" / "logs"
EXPECTED_RAW = PROJECT_ROOT.parent / "BCI Database"

if not RAW_PATH.is_symlink() or RAW_PATH.resolve(strict=True) != EXPECTED_RAW.resolve(strict=True):
    raise RuntimeError("data/raw must remain a relative symlink to the sibling BCI Database directory")

os.chdir(PROJECT_ROOT)
support_path = str(PROJECT_ROOT / "support")
if support_path not in sys.path:
    sys.path.insert(0, support_path)


## Phase implementation

In [2]:
"""Independently verify raw immutability and cleaned-file traceability."""

from __future__ import annotations

import sys
from pathlib import Path

from bci_core import (
    ARTICLE_URL,
    ISSUE_FIELDS,
    MANIFEST_FIELDS,
    Issue,
    assert_safe_paths,
    atomic_write_csv,
    atomic_write_text,
    common_parser,
    expected_participants,
    iter_files,
    log_event,
    merge_issues,
    new_run_id,
    normalize_frequency_csv_bytes,
    normalize_performances_csv_bytes,
    participant_id_from_path,
    read_csv_dicts,
    sha256_file,
)


POST_FIELDS = [
    "raw_relative_path",
    "cleaned_relative_path",
    "action",
    "raw_snapshot_sha256",
    "raw_current_sha256",
    "cleaned_current_sha256",
    "raw_unchanged",
    "cleaned_matches_expected",
    "semantic_equivalence",
    "status",
]


In [3]:
def main() -> int:
    parser = common_parser(__doc__ or "Verify cleaned dataset")
    args = parser.parse_args()
    raw, cleaned = assert_safe_paths(args.raw, args.cleaned)
    if not cleaned.is_dir():
        raise FileNotFoundError(f"Cleaned dataset does not exist: {cleaned}")
    inventory = read_csv_dicts(args.reports / "dataset_inventory.csv")
    manifest = read_csv_dicts(args.reports / "cleaning_manifest.csv")
    if not inventory or not manifest:
        raise RuntimeError("Inventory and cleaning manifest are required")
    manifest_by_raw = {row["raw_relative_path"]: row for row in manifest}
    if len(manifest_by_raw) != len(inventory):
        raise RuntimeError("Cleaning manifest does not contain exactly one entry per raw file")

    run_id = new_run_id()
    log_event(args.logs, run_id, "post_clean", "start", raw=str(raw), cleaned=str(cleaned))
    issues: list[Issue] = []
    results: list[dict[str, str]] = []
    updated_manifest: list[dict[str, str]] = []

    current_cleaned = {
        relative
        for path in iter_files(cleaned)
        if (relative := path.relative_to(cleaned).as_posix()) != ".gitkeep"
    }
    expected_cleaned = {
        row["cleaned_relative_path"] for row in manifest if row["cleaned_relative_path"]
    }
    extras = sorted(current_cleaned - expected_cleaned)
    missing = sorted(expected_cleaned - current_cleaned)
    for relative in extras:
        issues.append(
            Issue(
                "error", "post_clean", relative, participant_id_from_path(relative),
                "cleaned_membership", "unexpected file", "Only manifest-listed cleaned files may exist",
                ARTICLE_URL, "Remove only after confirming it was created by this pipeline.",
            )
        )
    for relative in missing:
        issues.append(
            Issue(
                "error", "post_clean", relative, participant_id_from_path(relative),
                "cleaned_membership", "missing file", "Every non-excluded raw file must be represented",
                ARTICLE_URL, "Rebuild the cleaned tree from the verified raw source.",
            )
        )

    for index, inventory_row in enumerate(inventory, start=1):
        relative = inventory_row["relative_path"]
        manifest_row = dict(manifest_by_raw[relative])
        raw_path = raw / relative
        raw_current = sha256_file(raw_path)
        raw_ok = raw_current == inventory_row["sha256"]
        action = manifest_row["action"]
        cleaned_relative = manifest_row["cleaned_relative_path"]
        cleaned_hash = ""
        cleaned_ok = False
        semantic = "not_applicable"

        if action == "excluded_administrative_artifact":
            cleaned_ok = not (cleaned / relative).exists()
            semantic = "raw-retained; cleaned-path-absent" if cleaned_ok else "failed"
        else:
            cleaned_path = cleaned / cleaned_relative
            if cleaned_path.is_file():
                cleaned_hash = sha256_file(cleaned_path)
                if action == "preserved_byte_identical":
                    cleaned_ok = cleaned_hash == inventory_row["sha256"]
                    semantic = "sha256-equal" if cleaned_ok else "failed"
                elif action == "normalized_frequency_csv":
                    expected_bytes, _ = normalize_frequency_csv_bytes(raw_path.read_bytes())
                    cleaned_ok = cleaned_path.read_bytes() == expected_bytes
                    semantic = "nonempty-cell-matrix-equal" if cleaned_ok else "failed"
                elif action == "normalized_performance_csv":
                    expected_bytes, _ = normalize_performances_csv_bytes(raw_path.read_bytes())
                    cleaned_ok = cleaned_path.read_bytes() == expected_bytes
                    semantic = "parsed-cell-matrix-equal" if cleaned_ok else "failed"

        status = "pass" if raw_ok and cleaned_ok else "fail"
        if not raw_ok:
            issues.append(
                Issue(
                    "fatal", "post_clean", relative, participant_id_from_path(relative),
                    "raw_immutability", raw_current, inventory_row["sha256"], ARTICLE_URL,
                    "Stop and restore the raw archive member; do not continue downstream.",
                )
            )
        if not cleaned_ok:
            issues.append(
                Issue(
                    "error", "post_clean", cleaned_relative or relative,
                    participant_id_from_path(relative), "cleaned_traceability", action,
                    "Cleaned file must satisfy its manifest action and equivalence rule", ARTICLE_URL,
                    "Rebuild the cleaned tree from the verified raw source.",
                )
            )
        if relative.lower().endswith(".gdf") and cleaned_hash != raw_current:
            issues.append(
                Issue(
                    "fatal", "post_clean", relative, participant_id_from_path(relative),
                    "gdf_byte_identity", cleaned_hash, raw_current, ARTICLE_URL,
                    "Discard the derived cleaned tree and rebuild; never repair the GDF.",
                )
            )

        results.append(
            {
                "raw_relative_path": relative,
                "cleaned_relative_path": cleaned_relative,
                "action": action,
                "raw_snapshot_sha256": inventory_row["sha256"],
                "raw_current_sha256": raw_current,
                "cleaned_current_sha256": cleaned_hash,
                "raw_unchanged": str(raw_ok).lower(),
                "cleaned_matches_expected": str(cleaned_ok).lower(),
                "semantic_equivalence": semantic,
                "status": status,
            }
        )
        manifest_row["cleaned_sha256"] = cleaned_hash
        manifest_row["equivalence_check"] = semantic
        manifest_row["status"] = "verified" if status == "pass" else "failed"
        updated_manifest.append(manifest_row)
        if index % 10 == 0 or index == len(inventory):
            log_event(args.logs, run_id, "post_clean", "hash_progress", completed=index, total=len(inventory))

    participant_dirs = {
        path.name
        for group in ("DATA A", "DATA B", "DATA C")
        for path in (cleaned / "Signals" / group).iterdir()
        if path.is_dir()
    }
    if participant_dirs != expected_participants():
        issues.append(
            Issue(
                "fatal", "post_clean", "Signals", "", "participant_id_set",
                f"count={len(participant_dirs)}", "Exact unchanged set of 87 participant IDs",
                ARTICLE_URL, "Rebuild the cleaned tree; do not rename participant directories.",
            )
        )

    atomic_write_csv(args.reports / "post_clean_validation.csv", results, POST_FIELDS)
    atomic_write_csv(args.reports / "cleaning_manifest.csv", updated_manifest, MANIFEST_FIELDS)
    merge_issues(args.reports / "validation_issues.csv", issues, {"post_clean"})
    failures = sum(row["status"] == "fail" for row in results) + len(extras) + len(missing)
    verdict = "PASS" if failures == 0 and not any(issue.severity in {"fatal", "error"} for issue in issues) else "FAIL"
    summary = [
        "# Post-clean Validation",
        "",
        f"- Result: **{verdict}**",
        f"- Raw files checked: {len(inventory):,}",
        f"- Cleaned files found: {len(current_cleaned):,}",
        f"- Unexpected cleaned files: {len(extras):,}",
        f"- Missing cleaned files: {len(missing):,}",
        f"- Failed traceability rows: {failures:,}",
        f"- Participant IDs preserved: {str(participant_dirs == expected_participants()).lower()}",
        "",
    ]
    atomic_write_text(args.reports / "post_clean_summary.md", "\n".join(summary))
    log_event(args.logs, run_id, "post_clean", "complete", result=verdict, failures=failures)
    print(f"Post-clean verification: {verdict}")
    return 0 if verdict == "PASS" else 2


## Execute phase

In [4]:
_arguments = [
    "05_verify.ipynb",
    "--raw", str(RAW_PATH),
    "--cleaned", str(CLEANED_PATH),
    "--reports", str(REPORTS_PATH),
    "--logs", str(LOGS_PATH),
]

_original_argv = sys.argv[:]
try:
    sys.argv = _arguments
    _exit_code = main()
finally:
    sys.argv = _original_argv

if _exit_code != 0:
    raise RuntimeError(f"Phase returned nonzero status {_exit_code}; inspect validation_issues.csv")


Post-clean verification: PASS


## Privacy-safe report check

In [5]:
import csv

report_path = REPORTS_PATH / "post_clean_validation.csv"
with report_path.open("r", encoding="utf-8", newline="") as handle:
    report_rows = list(csv.DictReader(handle))

{
    "report": str(report_path.relative_to(PROJECT_ROOT)),
    "rows": len(report_rows),
}


{'report': 'results/reports/post_clean_validation.csv', 'rows': 1073}